<a href="https://colab.research.google.com/github/pushplatadubey/collab-notebook/blob/master/A2Summ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up and Run A2Summ

This section automates cloning the repository, installing dependencies, compiling the custom CUDA kernels, and executing the summarization model.

In [ ]:
# 1. Clone the GitHub repository
!git clone https://github.com/boheumd/A2Summ.git
%cd A2Summ

In [ ]:
# 2. Install required environment dependencies
!pip install -r requirements.txt
# If requirements.txt is missing or needs specific versions, common packages include:
!pip install torch transformers tensorboardX rouge-score ortools gdown

In [ ]:
!pip install scipy h5py tensorboard

In [ ]:
# 3. Compile the C++/CUDA extension if needed by the project
# Navigating to any extension folder and running setup (typically under 'src/' or 'operators/' if applicable)
import os
if os.path.exists('operators'):
    %cd operators
    !python setup.py install
    %cd ..

## Dataset Setup

According to the repository, we need to download and set up the dataset (such as CNN/DailyMail or Gigaword) in the expected directory structure (usually a `data/` or `dataset/` folder inside the project).

In [ ]:
# Create dataset directories as needed by A2Summ
!mkdir -p data/cnn_dm
!mkdir -p dataset

### Official Dataset Download and Folder Structure

We will create the directory layout specified in the README and provide a download script using `gdown` to retrieve the dataset from Google Drive.

In [ ]:
# 1. Create the exact expected directory structure
!mkdir -p data/BLiSS/annotation data/BLiSS/feature
!mkdir -p data/CNN/annotation data/CNN/feature
!mkdir -p data/Daily_Mail/annotation data/Daily_Mail/feature
!mkdir -p data/SumMe/caption data/SumMe/feature
!mkdir -p data/TVSum/caption data/TVSum/feature

# Show the empty structure
!apt-get install -y tree > /dev/null
!tree data

In [ ]:
# 2. Download files from Google Drive folder link
# Folder link: https://drive.google.com/drive/folders/1rqXEIelRzq4mb7NaBk3GXxh7jlfP_Snm?usp=share_link
# Folder ID is '1rqXEIelRzq4mb7NaBk3GXxh7jlfP_Snm'
import gdown

folder_id = '1rqXEIelRzq4mb7NaBk3GXxh7jlfP_Snm'
try:
    gdown.download_folder(id=folder_id, output='data/', quiet=False, use_cookies=False)
except Exception as e:
    print(f"An error occurred during automatic folder download: {e}")
    print("If the folder download fails due to Drive restrictions, please manually download the files from Google Drive and upload them to their respective locations inside the '/content/A2Summ/data' folder.")

In [ ]:
# Extract the downloaded .tar archives and ensure correct structure
import os
import tarfile
import shutil

tar_dir = 'data/data'
dest_dir = 'data'

# 1. Perform clean extraction to parent directly
if os.path.exists(tar_dir):
    for filename in os.listdir(tar_dir):
        if filename.endswith('.tar'):
            tar_path = os.path.join(tar_dir, filename)
            print(f"Extracting {filename} directly into {dest_dir}...")
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=dest_dir)

    # Clean up archives
    shutil.rmtree(tar_dir, ignore_errors=True)
    print("Extraction complete.")

# 2. Fix the nested folders (e.g., moving contents of 'data/BLiSS/BLiSS' up to 'data/BLiSS')
for folder in ['BLiSS', 'CNN', 'Daily_Mail', 'SumMe', 'TVSum']:
    parent_path = os.path.join(dest_dir, folder)
    nested_path = os.path.join(parent_path, folder)
    if os.path.exists(nested_path):
        print(f"Correcting structure for {folder}...")
        # Move items out of the nested folder up to the parent folder
        for item in os.listdir(nested_path):
            src = os.path.join(nested_path, item)
            dst = os.path.join(parent_path, item)
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            shutil.move(src, dst)
        # Remove the now empty nested folder
        os.rmdir(nested_path)

# Verify the final verified structure
!tree data -L 3

In [ ]:
import os

helper_file = 'helpers/vsumm_helper.py'

# Ensure we start from a clean checkout
!git checkout -- {helper_file}

with open(helper_file, 'r') as f:
    lines = f.readlines()

new_lines = []
skip_until_def = False

for line in lines:
    if 'from ortools.algorithms.pywrapknapsack_solver import KnapsackSolver' in line:
        new_lines.append("from ortools.algorithms.python import knapsack_solver\n")
        continue

    if 'def knapsack(' in line:
        # Overwrite the knapsack function implementation directly to use the modern API
        knapsack_replacement = """def knapsack(values: Iterable[int], weights: Iterable[int], capacity: int) -> List[int]:
    # Modern ortools implementation
    solver = knapsack_solver.KnapsackSolver(
        knapsack_solver.SolverType.KNAPSACK_DYNAMIC_PROGRAMMING_SOLVER,
        'knapsack_solver'
    )
    # Convert float inputs if they are passed as floats
    values = [int(v) for v in values]
    weights = [int(w) for w in weights]
    capacity = int(capacity)

    solver.init(values, [weights], [capacity])
    computed_value = solver.solve()

    packed_items = []
    for i in range(len(values)):
        if solver.best_solution_contains(i):
            packed_items.append(i)
    return packed_items\n"""
        new_lines.append(knapsack_replacement)
        skip_until_def = True
        continue

    if skip_until_def:
        if line.startswith('def '):
            skip_until_def = False
        else:
            # Skip original knapsack body
            continue

    new_lines.append(line)

with open(helper_file, 'w') as f:
    f.writelines(new_lines)

print("Successfully replaced the knapsack function in vsumm_helper.py with a robust modern ortools wrapper!")

In [ ]:
!python train.py --dataset TVSum --batch_size 4 --max_epoch 5
# To run using CPU use following command
# !python train.py --dataset TVSum --device cpu --batch_size 4 --max_epoch 5

In [ ]:
mkdir saved_model;


In [ ]:
!tar -xvf /content/A2Summ/saved_model.tar


In [ ]:
!python train.py --dataset TVSum --test --checkpoint saved_model/TVSum --device cpu --batch_size 4 --max_epoch 5